In [ ]:
!pip install google-generativeai

In [ ]:

import google.generativeai as genai
import time
import json
import re

genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash")


def safe_generate(prompt, retries=3):
    for i in range(retries):
        try:
            return model.generate_content(prompt).text
        except Exception as e:
            print(f"Retry {i+1}: {e}")
            time.sleep(2)
    raise Exception("Gemini API failed after retries")



def parse_user_input(user_text):
    prompt = f"""
You are a smart scheduling assistant. Convert the following user input into structured JSON.

User input:
"{user_text}"

Return ONLY valid JSON in this exact format:
{{
  "tasks": [
    {{"name": "task name", "duration": <int minutes>, "priority": <int 1-3>, "splittable": <true/false>}}
  ],
  "fixed_events": [
    {{"name": "event name", "start": "HH:MM", "end": "HH:MM"}}
  ],
  "preferences": {{
    "wake_time": "<HH:MM or null>",
    "morning_routine_duration": <int minutes>,
    "break_duration": 10,
    "no_work_after": "22:00",
    "buffer_after_fixed": 10,
    "buffer_between_tasks": 5
  }}
}}

=== MORNING ROUTINE (SILENT — do NOT add it as a task or fixed event) ===
If user mentions wake time:
- Set wake_time to that time
- Estimate morning_routine_duration based on context:
  * Has class/work within 2 hrs of waking → 45 min (get ready, eat, commute prep)
  * No urgent event → 30 min
  * "quick morning" / "no breakfast" → 20 min
  * Mentions gym in morning → +60 min
- This is ONLY used to block out silent time. Never show it as a task.

=== TASK DURATION DEFAULTS ===
- assignment / homework / project = 120 min
- study / revision = 60 min
- go out / hangout / meet friends = 90 min
- lunch / dinner = 45 min
- gym = 60 min
- "quick X" → halve it; "long X" → +30 min

=== SPLITTABLE ===
- splittable = false for: assignments, focused study, gym, meals, outings (things that should NOT be broken up)
- splittable = true for: light reading, revision, chores (things okay to split across slots)

=== PRIORITY ===
- 3 = academic / deadline / exam
- 2 = personal errands, meals
- 1 = leisure, socializing

=== TIME PARSING ===
- "9 to 11:30", "9-11:30", "9;11:30" → start: "09:00", end: "11:30"
- Always 24-hour HH:MM

Output ONLY JSON. No markdown, no explanation.
"""

    raw = safe_generate(prompt)
    raw = re.sub(r"```(?:json)?", "", raw).strip()
    try:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if not match:
            raise ValueError("No JSON found")
        return json.loads(match.group())
    except Exception:
        print("Raw output:\n", raw)
        raise



def to_minutes(t):
    h, m = map(int, t.split(":"))
    return h * 60 + m

def to_time(m):
    return f"{m//60:02d}:{m%60:02d}"


def generate_schedule(data):
    tasks  = [dict(t) for t in data["tasks"]]
    fixed  = data["fixed_events"]
    prefs  = data["preferences"]

    end_day            = to_minutes(prefs.get("no_work_after", "22:00"))
    break_time         = prefs.get("break_duration", 10)
    buffer_after_fixed = prefs.get("buffer_after_fixed", 10)
    buffer_btw_tasks   = prefs.get("buffer_between_tasks", 5)

    # Silent morning routine — just moves start_day forward, never shown
    wake_raw    = prefs.get("wake_time", None)
    routine_dur = int(prefs.get("morning_routine_duration", 0))
    if wake_raw and wake_raw != "null" and routine_dur > 0:
        start_day = to_minutes(wake_raw) + routine_dur
    else:
        start_day = 8 * 60

    # Fixed event busy blocks
    busy = sorted([(to_minutes(e["start"]), to_minutes(e["end"]), e["name"]) for e in fixed])

    # Free slots (with buffer after each fixed event)
    free_slots = []
    current = start_day
    for s, e, _ in busy:
        if current < s:
            free_slots.append((current, s))
        current = max(current, e + buffer_after_fixed)
    if current < end_day:
        free_slots.append((current, end_day))

    # Sort: priority 3 → 2 → 1
    tasks.sort(key=lambda x: -x["priority"])

    schedule   = []
    task_index = 0

    for slot_start, slot_end in free_slots:
        cursor        = slot_start
        first_in_slot = True

        while task_index < len(tasks) and cursor < slot_end:
            task        = tasks[task_index]
            remaining   = task["duration"]
            splittable  = task.get("splittable", True)

            # Small buffer gap between tasks (not before first in slot)
            if not first_in_slot:
                if cursor + buffer_btw_tasks < slot_end:
                    cursor += buffer_btw_tasks  # silent gap, not shown as a block

            available = slot_end - cursor
            if available <= 0:
                break

            # NON-SPLITTABLE task: only schedule if it fits entirely in this slot
            if not splittable:
                if available >= remaining:
                    schedule.append({
                        "task":  task["name"],
                        "start": to_time(cursor),
                        "end":   to_time(cursor + remaining),
                        "type":  "task"
                    })
                    cursor       += remaining
                    task_index   += 1
                    first_in_slot = False
                else:
                    # Doesn't fit here — try next free slot
                    break

            else:
                while remaining > 0 and cursor < slot_end:
                    available = slot_end - cursor
                    if available <= 0:
                        break

                    work_block = min(remaining, available, 90)
                    schedule.append({
                        "task":  task["name"],
                        "start": to_time(cursor),
                        "end":   to_time(cursor + work_block),
                        "type":  "task"
                    })
                    cursor    += work_block
                    remaining -= work_block
                    first_in_slot = False

                    if remaining > 0 and cursor + break_time < slot_end:
                        schedule.append({
                            "task":  "Break",
                            "start": to_time(cursor),
                            "end":   to_time(cursor + break_time),
                            "type":  "break"
                        })
                        cursor += break_time

                task["duration"] = remaining
                if remaining <= 0:
                    task_index += 1

    # Fixed events for display
    fixed_display = [
        {"task": e["name"], "start": to_time(to_minutes(e["start"])),
         "end": to_time(to_minutes(e["end"])), "type": "fixed"}
        for e in fixed
    ]

    # Buffer after fixed events)
    buffer_display = []
    for s, e, _ in busy:
        buf_end = e + buffer_after_fixed
        if buf_end <= end_day:
            buffer_display.append({
                "task":  "—",
                "start": to_time(e),
                "end":   to_time(buf_end),
                "type":  "buffer"
            })

    all_blocks = fixed_display + buffer_display + schedule
    all_blocks.sort(key=lambda x: to_minutes(x["start"]))

    return all_blocks, tasks[task_index:]



def print_schedule(schedule, unscheduled):
    print("\n" + "─"*48)
    print("            YOUR DAY SCHEDULE")
    print("─"*48)

    icons = {"fixed": "", "task": "", "break": "", "buffer": "   "}

    prev_end = None
    for item in schedule:
        if item["type"] == "buffer":
            # Show buffer as a clean visual gap line, not a labelled block
            print(f"          ·  ({item['start']} – {item['end']}  free)")
            continue
        icon = icons.get(item.get("type", "task"), "▪")
        print(f"  {item['start']} – {item['end']}  {icon}  {item['task']}")

    if unscheduled:
        print("\n  ⚠️  Didn't fit today:")
        for t in unscheduled:
            print(f"     • {t['name']} ({t['duration']} min remaining)")

    print("─"*48)



user_input = input(" Enter your day plan:\n> ")
#print("\n⏳ Parsing...")
data = parse_user_input(user_input)
# parse output hidden
schedule, unscheduled = generate_schedule(data)
print_schedule(schedule, unscheduled)

 Enter your day plan:
> 3 hr classes from 9 , gym,hangout

────────────────────────────────────────────────
            YOUR DAY SCHEDULE
────────────────────────────────────────────────
  08:00 – 09:00    gym
  09:00 – 12:00    classes
          ·  (12:00 – 12:10  free)
  12:10 – 13:40    hangout
────────────────────────────────────────────────
